# Economic Structure vs Crime by District

This notebook is a **standalone test notebook** for connecting the economic data to district-level crime.

It is designed to work with the same project structure as your repo:

- `raw_data/economic_data.csv`
- `raw_data/socio_economic_data_2010.csv` *(optional, used for crime rate per 1,000 if available)*
- `processed_data/merged_weather_crime_districts_data.csv`

## What it does
- builds a district-day crime panel from the merged crime-weather file
- cleans the 2010 economic data
- extracts the two indicators you actually have:
  - **Total Employment**
  - **Average Annual Wage**
- merges them with district crime summaries
- shows:
  - crime vs average wage
  - crime vs total employment
  - optional crime rate per 1,000 population analysis
  - a ranked district comparison table


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path("..")
if not (BASE / "processed_data").exists():
    BASE = Path(".")

PROCESSED = BASE / "processed_data"
RAW = BASE / "raw_data"

crime_weather_path = PROCESSED / "merged_weather_crime_districts_data.csv"
econ_path = RAW / "economic_data.csv"
socio_path = RAW / "socio_economic_data_2010.csv"

print("Using base:", BASE.resolve())
print("Crime-weather file exists:", crime_weather_path.exists())
print("Economic file exists:", econ_path.exists())
print("Socio file exists:", socio_path.exists())

## Helper functions

In [ ]:
def normalize_colname(col):
    col = str(col).strip().replace("\xa0", " ")
    return re.sub(r"\s+", " ", col)

def clean_columns(df):
    out = df.copy()
    out.columns = [normalize_colname(c) for c in out.columns]
    return out

def first_existing(df, candidates, default=None):
    normalized = {str(c).lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in normalized:
            return normalized[cand.lower()]
    return default

def parse_district_value(x):
    if pd.isna(x):
        return np.nan
    m = re.search(r"(\d+)", str(x))
    return float(m.group(1)) if m else np.nan

def clean_us_number(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.strip()
        .replace({"nan": np.nan, "None": np.nan, "": np.nan}),
        errors="coerce"
    )

def add_district_labels(ax, x, y, labels, fontsize=8):
    for xi, yi, lab in zip(x, y, labels):
        if pd.notna(xi) and pd.notna(yi):
            ax.annotate(str(int(lab)), (xi, yi), textcoords="offset points", xytext=(4, 4), fontsize=fontsize)

def safe_polyfit_line(ax, x, y):
    mask = pd.notna(x) & pd.notna(y)
    if mask.sum() >= 2:
        z = np.polyfit(x[mask], y[mask], 1)
        p = np.poly1d(z)
        xr = np.linspace(x[mask].min(), x[mask].max(), 100)
        ax.plot(xr, p(xr), linewidth=2, linestyle="--")

## 1. Load and aggregate crime data

In [ ]:
crime_weather = clean_columns(pd.read_csv(crime_weather_path))

district_col = first_existing(
    crime_weather,
    ["district", "District", "COUNCIL DISTRICT", "Council District", "council_district", "district_id", "CD", "cd"]
)
datetime_col = first_existing(
    crime_weather,
    ["DATETIME OCC", "datetime occ", "datetime", "DATE OCC", "DATE RPTD", "date"]
)
crime_col = first_existing(
    crime_weather,
    ["CRIME", "crime", "Crm Cd Desc", "crime_type"]
)

print("Detected district column:", district_col)
print("Detected datetime column:", datetime_col)
print("Detected crime column:", crime_col)

if district_col is None or datetime_col is None:
    raise ValueError("Could not detect district/date columns in merged_weather_crime_districts_data.csv")

cw = crime_weather.copy()
cw[datetime_col] = pd.to_datetime(cw[datetime_col], errors="coerce")
cw[district_col] = cw[district_col].apply(parse_district_value).astype("Int64")
cw["date"] = cw[datetime_col].dt.floor("D")
cw = cw.dropna(subset=["date", district_col]).copy()

panel = (
    cw.groupby([district_col, "date"], as_index=False)
      .agg({crime_col: "count"} if crime_col is not None else {})
      .rename(columns={district_col: "district", crime_col: "crime_count"})
)

if "crime_count" not in panel.columns:
    panel["crime_count"] = 1

panel["district"] = panel["district"].astype("Int64")
panel = panel.sort_values(["district", "date"]).reset_index(drop=True)

panel.head()

In [ ]:
district_crime = (
    panel.groupby("district")
    .agg(
        avg_daily_crime=("crime_count", "mean"),
        total_crime=("crime_count", "sum"),
        days_observed=("date", "nunique")
    )
    .reset_index()
    .sort_values("avg_daily_crime", ascending=False)
)

district_crime

## 2. Load and clean economic data (2010 only)

In [ ]:
economic_data = clean_columns(pd.read_csv(econ_path))

economic_data["council_district"] = pd.to_numeric(economic_data["council_district"], errors="coerce")
economic_data = economic_data[economic_data["council_district"].isin(range(1, 16))].copy()
economic_data = economic_data[economic_data["cy_qtr"].isin(["2010-01", "2010-02", "2010-03", "2010-04"])].copy()

economic_data["value_num"] = clean_us_number(economic_data["value"])

print("Indicators available:")
print(economic_data["indicator"].dropna().unique())

economic_data.head()

In [ ]:
wage_by_district = (
    economic_data[economic_data["indicator"] == "Average Annual Wage"]
    .groupby("council_district", as_index=False)["value_num"]
    .mean()
    .rename(columns={"value_num": "avg_annual_wage"})
)

employment_by_district = (
    economic_data[economic_data["indicator"] == "Total Employment"]
    .groupby("council_district", as_index=False)["value_num"]
    .mean()
    .rename(columns={"value_num": "total_employment"})
)

wage_by_district, employment_by_district.head()

## 3. Optional population data for crime rate per 1,000

In [ ]:
pop_by_district = None

if socio_path.exists():
    socio = clean_columns(pd.read_csv(socio_path))
    socio_district_col = first_existing(
        socio,
        ["Council District", "council_district", "district", "CD #", "CD#", "CD"]
    )
    if socio_district_col is not None:
        socio["district"] = socio[socio_district_col].apply(parse_district_value).astype("Int64")
        if "Pop2010" in socio.columns:
            socio["Pop2010"] = clean_us_number(socio["Pop2010"])
            pop_by_district = socio[["district", "Pop2010"]].dropna().drop_duplicates("district")
            print("Population data loaded.")
        else:
            print("Pop2010 not found in socio data.")
    else:
        print("District column not found in socio data.")
else:
    print("No socio data file found; rate-per-1000 section will be skipped.")

pop_by_district

## 4. Merge crime + economic data

In [ ]:
district_econ_crime = (
    district_crime
    .merge(wage_by_district, left_on="district", right_on="council_district", how="left")
    .merge(employment_by_district, left_on="district", right_on="council_district", how="left", suffixes=("_wage", "_employment"))
)

drop_cols = [c for c in district_econ_crime.columns if "council_district" in c]
district_econ_crime = district_econ_crime.drop(columns=drop_cols)

if pop_by_district is not None:
    district_econ_crime = district_econ_crime.merge(pop_by_district, on="district", how="left")
    district_econ_crime["crime_per_1000"] = np.where(
        district_econ_crime["Pop2010"] > 0,
        district_econ_crime["total_crime"] / district_econ_crime["Pop2010"] * 1000,
        np.nan
    )

district_econ_crime = district_econ_crime.sort_values("avg_daily_crime", ascending=False).reset_index(drop=True)
district_econ_crime

## 5. Quick correlation check

In [ ]:
corr_cols = ["avg_daily_crime", "avg_annual_wage", "total_employment"]
if "crime_per_1000" in district_econ_crime.columns:
    corr_cols = ["avg_daily_crime", "crime_per_1000", "avg_annual_wage", "total_employment"]

district_econ_crime[corr_cols].corr(numeric_only=True)

## 6. Crime vs average annual wage

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(district_econ_crime["avg_annual_wage"], district_econ_crime["avg_daily_crime"], s=80, alpha=0.8)
add_district_labels(
    ax,
    district_econ_crime["avg_annual_wage"],
    district_econ_crime["avg_daily_crime"],
    district_econ_crime["district"]
)
safe_polyfit_line(ax, district_econ_crime["avg_annual_wage"], district_econ_crime["avg_daily_crime"])
ax.set_xlabel("Average Annual Wage")
ax.set_ylabel("Average Daily Crime")
ax.set_title("Average Daily Crime vs Average Annual Wage by District")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Crime vs total employment

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(district_econ_crime["total_employment"], district_econ_crime["avg_daily_crime"], s=80, alpha=0.8)
add_district_labels(
    ax,
    district_econ_crime["total_employment"],
    district_econ_crime["avg_daily_crime"],
    district_econ_crime["district"]
)
safe_polyfit_line(ax, district_econ_crime["total_employment"], district_econ_crime["avg_daily_crime"])
ax.set_xlabel("Total Employment")
ax.set_ylabel("Average Daily Crime")
ax.set_title("Average Daily Crime vs Total Employment by District")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Optional rate-based versions

In [ ]:
if "crime_per_1000" in district_econ_crime.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].scatter(district_econ_crime["avg_annual_wage"], district_econ_crime["crime_per_1000"], s=80, alpha=0.8)
    add_district_labels(
        axes[0],
        district_econ_crime["avg_annual_wage"],
        district_econ_crime["crime_per_1000"],
        district_econ_crime["district"]
    )
    safe_polyfit_line(axes[0], district_econ_crime["avg_annual_wage"], district_econ_crime["crime_per_1000"])
    axes[0].set_xlabel("Average Annual Wage")
    axes[0].set_ylabel("Crime per 1,000 residents")
    axes[0].set_title("Crime Rate vs Wage")
    axes[0].grid(alpha=0.3)

    axes[1].scatter(district_econ_crime["total_employment"], district_econ_crime["crime_per_1000"], s=80, alpha=0.8)
    add_district_labels(
        axes[1],
        district_econ_crime["total_employment"],
        district_econ_crime["crime_per_1000"],
        district_econ_crime["district"]
    )
    safe_polyfit_line(axes[1], district_econ_crime["total_employment"], district_econ_crime["crime_per_1000"])
    axes[1].set_xlabel("Total Employment")
    axes[1].set_ylabel("Crime per 1,000 residents")
    axes[1].set_title("Crime Rate vs Employment")
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("crime_per_1000 not available because Pop2010 was not found.")

## 9. Ranked comparison tables

In [ ]:
display_cols = ["district", "avg_daily_crime", "total_crime", "avg_annual_wage", "total_employment"]
if "crime_per_1000" in district_econ_crime.columns:
    display_cols.insert(2, "crime_per_1000")

district_econ_crime[display_cols].sort_values("avg_daily_crime", ascending=False).reset_index(drop=True)

In [ ]:
high_wage = district_econ_crime.nlargest(5, "avg_annual_wage")
low_wage = district_econ_crime.nsmallest(5, "avg_annual_wage")
high_employment = district_econ_crime.nlargest(5, "total_employment")
low_employment = district_econ_crime.nsmallest(5, "total_employment")

print("Average daily crime in top 5 wage districts:", round(high_wage["avg_daily_crime"].mean(), 2))
print("Average daily crime in bottom 5 wage districts:", round(low_wage["avg_daily_crime"].mean(), 2))
print("Average daily crime in top 5 employment districts:", round(high_employment["avg_daily_crime"].mean(), 2))
print("Average daily crime in bottom 5 employment districts:", round(low_employment["avg_daily_crime"].mean(), 2))

## How to read the results

- If **crime rises with total employment**, that can mean busier districts attract more daily activity and more opportunity-based crime.
- If **crime falls as wages rise**, that supports a socioeconomic gradient.
- If relationships are weak, that is still useful: it means economic structure alone does not explain district crime very well.

This notebook is intentionally **district-level**, because your economic notebook is using **2010 structural indicators**, not a full time-aligned economic series for 2010–2019.
